# Test MCP Tools

Test each MCP server tool independently to verify correct operation before integrating with the harness.

## Prerequisites
- PostgreSQL with pgvector running (`make dev-up`)
- Backend started (`make backend-start` — auto-starts MCP servers)
- At least one document ingested (see `1_document_processing/`)
- LLM and embedding endpoints configured in `.env`

In [ ]:
import os
import sys
import json
import httpx
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from dotenv import load_dotenv
env_path = os.path.join('..', '.env')
load_dotenv(env_path, override=True)

VECTOR_SEARCH_MCP_URL = os.getenv("VECTOR_SEARCH_MCP_URL", "http://127.0.0.1:9002")
WEB_SEARCH_MCP_URL = os.getenv("WEB_SEARCH_MCP_URL", "http://127.0.0.1:9003")
VERIFICATION_MCP_URL = os.getenv("VERIFICATION_MCP_URL", "http://127.0.0.1:9004")
OBSERVABILITY_MCP_URL = os.getenv("OBSERVABILITY_MCP_URL", "http://127.0.0.1:9005")

print("MCP Server URLs:")
print(f"  vector-search-mcp: {VECTOR_SEARCH_MCP_URL}")
print(f"  web-search-mcp:    {WEB_SEARCH_MCP_URL}")
print(f"  verification-mcp:  {VERIFICATION_MCP_URL}")
print(f"  observability-mcp: {OBSERVABILITY_MCP_URL}")

## 1. Health Check — Verify MCP Servers Are Running

In [ ]:
servers = {
    "vector-search-mcp": VECTOR_SEARCH_MCP_URL,
    "web-search-mcp": WEB_SEARCH_MCP_URL,
    "verification-mcp": VERIFICATION_MCP_URL,
    "observability-mcp": OBSERVABILITY_MCP_URL,
}

for name, url in servers.items():
    try:
        resp = httpx.post(
            f"{url}/mcp",
            json={"jsonrpc": "2.0", "method": "tools/list", "id": 1},
            timeout=5.0,
        )
        if resp.status_code == 200:
            tools = resp.json().get("result", {}).get("tools", [])
            print(f"  \u2705 {name}: {len(tools)} tools available")
        else:
            print(f"  \u274c {name}: HTTP {resp.status_code}")
    except Exception as e:
        print(f"  \u274c {name}: {e}")

## 2. Test Vector Search MCP

In [ ]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client
import asyncio

async def test_vector_search():
    async with streamablehttp_client(f"{VECTOR_SEARCH_MCP_URL}/mcp") as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool("semantic_search", {
                "query": "machine learning",
                "top_k": 3,
            })
            data = json.loads(result.content[0].text)
            print(f"\u2705 semantic_search returned {len(data)} results")
            for r in data[:2]:
                print(f"   - {r.get('document_name', 'N/A')} (sim: {r.get('similarity', 0):.3f})")

await test_vector_search()

## 3. Test Web Search MCP

In [ ]:
async def test_web_search():
    async with streamablehttp_client(f"{WEB_SEARCH_MCP_URL}/mcp") as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool("web_search", {
                "query": "Red Hat OpenShift AI",
                "num_results": 3,
            })
            data = json.loads(result.content[0].text)
            print(f"\u2705 web_search returned {len(data)} results")
            for r in data:
                print(f"   - {r.get('title', 'N/A')}")
                print(f"     {r.get('url', '')}")

await test_web_search()

## 4. Test Verification MCP

In [ ]:
async def test_verification():
    async with streamablehttp_client(f"{VERIFICATION_MCP_URL}/mcp") as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool("quality_score", {
                "draft": "This is a sample research report about AI.",
                "query": "What is AI?",
            })
            data = json.loads(result.content[0].text)
            print(f"\u2705 quality_score returned:")
            print(f"   Overall: {data.get('overall', 'N/A')}/10")
            print(f"   Feedback: {data.get('feedback', 'N/A')}")

await test_verification()

## 5. Test Observability MCP

In [ ]:
async def test_observability():
    async with streamablehttp_client(f"{OBSERVABILITY_MCP_URL}/mcp") as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool("get_metrics", {
                "session_id": "test-session-001",
            })
            data = json.loads(result.content[0].text)
            print(f"\u2705 get_metrics returned: {json.dumps(data, indent=2)[:200]}")

await test_observability()

## Summary

All MCP servers verified:
- **vector-search-mcp** (9002) — Semantic search over pgvector
- **web-search-mcp** (9003) — Real-time web search via SearXNG
- **verification-mcp** (9004) — Quality scoring and fact-checking
- **observability-mcp** (9005) — Trace and failure logging

Document ingestion is handled directly by the backend API (Docling → embedding → pgvector).

Next: Harness Engineering (Phase 3) →